<a href="https://colab.research.google.com/github/shaan-byte/python_ds_colab/blob/main/NumPy_Arrays_and_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 5.1 — NumPy Arrays & Performance

**Module 2: Data Analysis Foundations**

---

## What we will cover today

1. Why NumPy exists — and what problem it solves
2. Creating NumPy arrays
3. Array properties — shape, dtype, ndim, size
4. N-dimensional arrays — 1D, 2D, and 3D
5. Array operations — why they are fast
6. Performance comparison — NumPy vs Python lists

---

> **The thread today:** We are analysts for an IPL franchise. We have match-by-match run data for our team across an entire season. Everything we do today — creating arrays, reshaping them, understanding their properties, measuring performance — will be grounded in that dataset.

---
## Setup — Run this first

In [ ]:
import numpy as np
import time

print("NumPy version:", np.__version__)
print("Ready.")

NumPy version: 2.0.2
Ready.


---
# Part 1 — Why Does NumPy Exist?

## The problem with Python lists for numbers

Python lists are flexible and convenient. But they have a serious limitation when it comes to numeric computation: **they were not designed for it**.

Consider this: you have the runs scored by a batsman in every match of the IPL season. You want to find:
- The total runs
- The average
- How many matches they scored above 50
- All scores multiplied by 1.1 (a hypothetical bonus calculation)

With a Python list, you have to write loops for all of these. Let's see how that looks:

In [ ]:
# Virat's runs per match this IPL season (14 matches)
virat_runs_list = [72, 45, 88, 31, 64, 0, 103, 52, 19, 77, 41, 93, 28, 66]

# Total runs — need a loop or sum()
# total = sum(virat_runs_list)
total = 0
for score in virat_runs_list:
  total = total + score

# Average — another calculation
average = total / len(virat_runs_list)

# Matches above 50 — need a loop
above_50 = [score for score in virat_runs_list if score > 50]

# Multiply every score by 1.1 — ANOTHER loop
bonus_scores = [score * 1.1 for score in virat_runs_list]

print(f"Total:       {total}")
print(f"Average:     {average:.1f}")
print(f"Above 50:    {above_50}")
print(f"Bonus scores (first 5): {bonus_scores[:5]}")
print()
print("Every operation needed a loop or a list comprehension.")
print("Now imagine doing this on 100,000 rows of data.")

Total:       779
Average:     55.6
Above 50:    [72, 88, 64, 103, 52, 77, 93, 66]
Bonus scores (first 5): [79.2, 49.50000000000001, 96.80000000000001, 34.1, 70.4]

Every operation needed a loop or a list comprehension.
Now imagine doing this on 100,000 rows of data.


### The NumPy answer

NumPy was built specifically for numeric computation on arrays of data. It solves the loop problem in two ways:

**1. Vectorisation** — operations apply to the whole array at once, without writing a loop. Internally, NumPy runs optimised C code that processes every element.

**2. Homogeneous storage** — a NumPy array stores all elements as the same type in a contiguous block of memory, like a C array. A Python list stores pointers to individual objects scattered in memory. The NumPy layout is far faster for bulk operations.

Let's see the same four operations with NumPy:

In [ ]:
diff_list = [1, '123', True]

In [ ]:
diff_list

[1, '123', True]

In [ ]:
# The same data as a NumPy array
virat_runs = np.array(virat_runs_list)

# All four operations — no loops anywhere
total        = virat_runs.sum()
average      = virat_runs.mean()
above_50     = virat_runs[virat_runs > 50]    # boolean indexing — covered soon
bonus_scores = virat_runs * 1.1               # applied to every element at once

print(f"Total:       {total}")
print(f"Average:     {average:.1f}")
print(f"Above 50:    {above_50}")
print(f"Bonus scores (first 5): {bonus_scores[:5]}")
print()
print("Same results. No explicit loops. And much faster on large data.")

Total:       779
Average:     55.6
Above 50:    [ 72  88  64 103  52  77  93  66]
Bonus scores (first 5): [79.2 49.5 96.8 34.1 70.4]

Same results. No explicit loops. And much faster on large data.


In [ ]:
new_arr = np.array([132, "123"], dtype='int')

In [ ]:
new_arr

array([132, 123])

### Whiteboard: how memory layout explains the speed difference

```
Python list — [72, 45, 88, 31, ...]

  Memory:  [ptr] [ptr] [ptr] [ptr] ...
              |     |     |     |
           [obj] [obj] [obj] [obj]   <- each number is a full Python object
                                        stored somewhere in memory
           Each access follows a pointer. Slow for bulk math.

NumPy array — np.array([72, 45, 88, 31, ...])

  Memory:  [72][45][88][31][64][0][103]...
            ^--- all numbers stored back-to-back, same type, no pointers ---^
           The CPU can load and process many at once. Very fast for bulk math.
```

This is the core reason NumPy is fast: **contiguous, typed memory** versus **scattered pointers**.

In [ ]:
a = [1, 23123, 123, 123,  122]

In [ ]:
a.append(123445)

In [ ]:
print(a)

[1, 23123, 123, 123, 122, 123445]


---
# Part 2 — Creating NumPy Arrays

There are several ways to create a NumPy array. They fall into two categories:
- **From existing data** — convert a Python list or other sequence
- **From scratch** — NumPy generates the values for you

In [ ]:
# Method 1: From a Python list
rohit_runs = np.array([56, 0, 74, 88, 32, 45, 61, 17, 90, 38, 72, 5, 83, 49])
print("From list:")
print(rohit_runs)
print(type(rohit_runs))
print()

From list:
[56  0 74 88 32 45 61 17 90 38 72  5 83 49]
<class 'numpy.ndarray'>



In [ ]:
# Method 2: np.zeros() — array of all zeros
# Useful when you want to pre-allocate space before filling it in
placeholder = np.zeros(14)      # 14 match slots, all set to 0.0
print("np.zeros(14):")
print(placeholder)
print()

# Method 3: np.ones() — array of all ones
ones = np.ones(5)
print("np.ones(5):")
print(ones)
print()

# Method 4: np.arange() — evenly spaced values (like Python's range())
# Match numbers 1 through 14
match_numbers = np.arange(1, 15)   # start=1, stop=15 (exclusive)
print("Match numbers np.arange(1, 15):")
print(match_numbers)
print()

# Method 5: np.linspace() — N evenly spaced values between start and stop
# Useful for generating test data, plots, or smooth ranges
run_rate_scale = np.linspace(6.0, 12.0, num=7)   # 7 points from 6.0 to 12.0
print("np.linspace(6.0, 12.0, 7):")
print(run_rate_scale)
print()

# Method 6: np.full() — array filled with a specific value
duck_out = np.full(5, fill_value=0)   # 5 ducks in a row!
print("np.full(5, fill_value=0):")
print(duck_out)

np.zeros(14):
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

np.ones(5):
[1. 1. 1. 1. 1.]

Match numbers np.arange(1, 15):
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14]

np.linspace(6.0, 12.0, 7):
[ 6.  7.  8.  9. 10. 11. 12.]

np.full(5, fill_value=0):
[0 0 0 0 0]


In [ ]:
placeholder.shape

(14,)

In [ ]:
two_d = np.array([
    [1, 2, 3],
    [4, 5, 6]
  ])

In [ ]:
two_d.shape

(2, 3)

In [ ]:
np.arange(0, 10, 2)

array([0, 2, 4, 6, 8])

In [ ]:
np.linspace(6, 12, num=14)

array([ 6.        ,  6.46153846,  6.92307692,  7.38461538,  7.84615385,
        8.30769231,  8.76923077,  9.23076923,  9.69230769, 10.15384615,
       10.61538462, 11.07692308, 11.53846154, 12.        ])

In [ ]:
np.full(5, fill_value='sriram')

array(['sriram', 'sriram', 'sriram', 'sriram', 'sriram'], dtype='<U6')

---
### Exercise 1 — Creating Arrays

**Task:** Create four arrays as described. Use the correct NumPy function for each.

In [ ]:
# 1. Hardik's runs per match this season
hardik_runs = np.array([___, ___, ___, ___, ___, ___, ___, ___, ___, ___, ___, ___, ___, ___])
# <-- fill in 14 numbers of your choice (realistic cricket scores: 0-120)

# 2. An array of zeros representing 14 empty bowling slots
bowling_slots = np.zeros(14)      # <-- which function? how many?

# 3. Over numbers 1 through 20 (for a T20 match)
over_numbers = np.arange(1, 21)  # <-- which function? what start and stop?

# 4. 5 evenly spaced economy rates between 6.5 and 9.5
economy_scale = np.linspace(6.5,9.5, num=5)  # <-- which function? start, stop, count?


print("hardik_runs:   ", hardik_runs)
print("bowling_slots: ", bowling_slots)
print("over_numbers:  ", over_numbers)
print("economy_scale: ", economy_scale)
print()
print(f"Correct (14 hardik scores):      {len(hardik_runs) == 14}")
print(f"Correct (14 zero slots):         {len(bowling_slots) == 14 and bowling_slots.sum() == 0}")
print(f"Correct (overs 1-20):            {list(over_numbers) == list(range(1, 21))}")
print(f"Correct (5 economy values):      {len(economy_scale) == 5}")
print(f"Correct (economy start/end):     {economy_scale[0] == 6.5 and economy_scale[-1] == 9.5}")

<details>
<summary>Show solution</summary>

```python
hardik_runs   = np.array([45, 32, 0, 67, 18, 54, 78, 22, 41, 33, 60, 15, 88, 27])
bowling_slots = np.zeros(14)
over_numbers  = np.arange(1, 21)
economy_scale = np.linspace(6.5, 9.5, num=5)
```

</details>

---
# Part 3 — Array Properties

Every NumPy array carries metadata that describes its structure. These are not methods you call with `()` — they are **attributes** you access directly, like `array.shape`.

Understanding these properties is essential before working with multi-dimensional arrays, because they tell you exactly what you are dealing with.

| Property | What it tells you | Example |
|---|---|---|
| `.shape` | Dimensions as a tuple | `(14,)` or `(3, 14)` |
| `.ndim` | Number of dimensions | `1`, `2`, `3` |
| `.size` | Total number of elements | `14`, `42` |
| `.dtype` | Data type of elements | `int64`, `float64` |
| `.nbytes` | Memory used in bytes | `112` |

In [ ]:
# Inspect the properties of virat_runs
virat_runs = np.array([72, 45, 88, 31, 64, 0, 103, 52, 19, 77, 41, 93, 28, 66])

print("Array:", virat_runs)
print()
print(f"  .shape  = {virat_runs.shape}   <- dimensions as a tuple")
print(f"  .ndim   = {virat_runs.ndim}         <- number of dimensions")
print(f"  .size   = {virat_runs.size}        <- total number of elements")
print(f"  .dtype  = {virat_runs.dtype}       <- data type of each element")
print(f"  .nbytes = {virat_runs.nbytes}       <- bytes used in memory")
print()
print(f"Bytes per element: {virat_runs.nbytes // virat_runs.size} (int64 = 8 bytes)")

Array: [ 72  45  88  31  64   0 103  52  19  77  41  93  28  66]

  .shape  = (14,)   <- dimensions as a tuple
  .ndim   = 1         <- number of dimensions
  .size   = 14        <- total number of elements
  .dtype  = int64       <- data type of each element
  .nbytes = 112       <- bytes used in memory

Bytes per element: 8 (int64 = 8 bytes)


In [ ]:
# dtype — the data type — is important to understand
# NumPy infers it from the data you provide

int_array    = np.array([72, 45, 88])           # integers -> int64
float_array  = np.array([72.0, 45.5, 88.3])     # floats -> float64
mixed_array  = np.array([72, 45.5, 88])         # mixed -> float64 (upcasted)
bool_array   = np.array([True, False, True])     # booleans -> bool

for name, arr in [("int_array", int_array), ("float_array", float_array),
                  ("mixed_array", mixed_array),  ("bool_array", bool_array)]:
    print(f"  {name:<15} dtype={arr.dtype:<10} values={arr}")

print()
print("Key rule: a NumPy array can only hold ONE type.")
print("If you mix int and float, NumPy upcasts everything to float.")
print("If you mix numbers and strings, everything becomes a string.")
print()

# You can also specify the dtype explicitly
run_rate = np.array([8, 7, 9, 6], dtype=float)
print("Explicit dtype=float:", run_rate, "  dtype:", run_rate.dtype)

In [ ]:
# The .shape tuple is the most important property to understand
# Its length tells you ndim, and each value tells you the size of that dimension

a1 = np.array([1, 2, 3, 4, 5])           # 1D: 5 elements
a2 = np.array([[1, 2, 3], [4, 5, 6]])    # 2D: 2 rows, 3 columns
a3 = np.zeros((2, 3, 4))                 # 3D: 2 blocks, 3 rows, 4 columns

for name, arr in [("a1 (1D)", a1), ("a2 (2D)", a2), ("a3 (3D)", a3)]:
    print(f"  {name:<12}  shape={str(arr.shape):<12} ndim={arr.ndim}  size={arr.size}")

print()
print("Reading .shape:")
print(f"  a2.shape = (2, 3) means: 2 rows, 3 columns -> 6 elements total")
print(f"  a3.shape = (2, 3, 4) means: 2 blocks, each with 3 rows and 4 columns -> 24 elements")

---
### Exercise 2 — Array Properties

**Task:** Answer the questions in the comments by reading the properties of each array. Replace each `None` with the correct expression — do not just type the answer, write the code that gives it.

In [ ]:
match_scores = np.array([
    [72, 45, 88, 31, 64, 0,   103, 52],   # Virat — first 8 matches
    [56, 0,  74, 88, 32, 45,  61,  17],   # Rohit — first 8 matches
    [45, 32, 0,  67, 18, 54,  78,  22],   # Hardik — first 8 matches
])

# Q1: How many dimensions does match_scores have?
q1 = match_scores.ndim

# Q2: What is the shape?
q2 = match_scores.shape

# Q3: How many total scores are stored?
q3 = match_scores.size

# Q4: What data type are the elements?
q4 = match_scores.dtype

# Q5: How many bytes does the array use in memory?
q5 = match_scores.nbyte

# Q6: How many rows (players) are there?
#     Hint: .shape is a tuple — index into it
q6 = match_scores.shape[0]

# Q7: How many columns (matches) are there?
q7 = match_scores.shape[1]


print(f"Q1 ndim:       {q1}")
print(f"Q2 shape:      {q2}")
print(f"Q3 size:       {q3}")
print(f"Q4 dtype:      {q4}")
print(f"Q5 nbytes:     {q5}")
print(f"Q6 num players:{q6}")
print(f"Q7 num matches:{q7}")
print()
print(f"Correct: {q1==2 and q2==(3,8) and q3==24 and q6==3 and q7==8}")

<details>
<summary>Show solution</summary>

```python
q1 = match_scores.ndim
q2 = match_scores.shape
q3 = match_scores.size
q4 = match_scores.dtype
q5 = match_scores.nbytes
q6 = match_scores.shape[0]
q7 = match_scores.shape[1]
```

</details>

---
# Part 4 — N-Dimensional Arrays

## Why more than one dimension?

A 1D array is a single sequence — like one player's scores across matches. But real data is almost always tabular — rows and columns. That is a 2D array.

And some data is naturally 3D:
- An image is 2D (height x width), but a colour image is 3D (height x width x 3 colour channels)
- Our cricket data could be 3D: (seasons x players x matches)

NumPy handles all of these with the same set of tools.

---

## 1D — A single player's scores

In [ ]:
# 1D array: Virat's runs across 14 matches
virat_runs = np.array([72, 45, 88, 31, 64, 0, 103, 52, 19, 77, 41, 93, 28, 66])

print("1D Array — Virat's runs:")
print(virat_runs)
print(f"Shape: {virat_runs.shape}  ->  {virat_runs.shape[0]} matches")
print()

# Indexing: same as a Python list
print(f"Match 1 score (index 0):    {virat_runs[0]}")
print(f"Last match score (index -1):{virat_runs[-1]}")
print(f"First 5 matches:            {virat_runs[:5]}")
print(f"Matches 3 to 7:             {virat_runs[2:7]}")

1D Array — Virat's runs:
[ 72  45  88  31  64   0 103  52  19  77  41  93  28  66]
Shape: (14,)  ->  14 matches

Match 1 score (index 0):    72
Last match score (index -1):66
First 5 matches:            [72 45 88 31 64]
Matches 3 to 7:             [ 88  31  64   0 103]


## 2D — The whole team's scores

A 2D array is like a table. Think of it as **rows and columns**.

In our case: rows = players, columns = matches.

```
              Match1  Match2  Match3  Match4  ...
Virat   row 0:   72      45      88      31   ...
Rohit   row 1:   56       0      74      88   ...
Hardik  row 2:   45      32       0      67   ...
```

In [ ]:
# 2D array: 3 players x 14 matches
team_scores = np.array([
    [72, 45, 88, 31, 64,  0, 103, 52, 19, 77, 41, 93, 28, 66],  # row 0: Virat
    [56,  0, 74, 88, 32, 45,  61, 17, 90, 38, 72,  5, 83, 49],  # row 1: Rohit
    [45, 32,  0, 67, 18, 54,  78, 22, 41, 33, 60, 15, 88, 27],  # row 2: Hardik
])

print("2D Array — team_scores:")
print(team_scores)
print()
print(f"Shape: {team_scores.shape}  ->  {team_scores.shape[0]} players, {team_scores.shape[1]} matches")

2D Array — team_scores:
[[ 72  45  88  31  64   0 103  52  19  77  41  93  28  66]
 [ 56   0  74  88  32  45  61  17  90  38  72   5  83  49]
 [ 45  32   0  67  18  54  78  22  41  33  60  15  88  27]]

Shape: (3, 14)  ->  3 players, 14 matches


In [ ]:
a = [
    [1, 2],
    [3, 4]
]

a[0][1]

2

In [ ]:
# Indexing a 2D array: [row, column]
# Think of it as [player_index, match_index]

print("2D Indexing examples:")
print(f"  team_scores[0, 0]   = {team_scores[0, 0]}  (Virat, match 1)")
print(f"  team_scores[1, 2]   = {team_scores[1, 2]}  (Rohit, match 3)")
print(f"  team_scores[2, -1]  = {team_scores[2, -1]}  (Hardik, last match)")
print()

# Slicing a whole row (one player's entire season)
print(f"  team_scores[0, :]   = {team_scores[0, :]}")
print(f"  (Virat's scores all season)")
print()

# Slicing a whole column (all players' scores in one match)
print(f"  team_scores[:, 0]   = {team_scores[:, 0]}")
print(f"  (All players' scores in match 1)")
print()

# Slicing a sub-region
print(f"  team_scores[0:2, 0:4] =  (first 2 players, first 4 matches)")
print(team_scores[0:2, 0:4])

2D Indexing examples:
  team_scores[0, 0]   = 72  (Virat, match 1)
  team_scores[1, 2]   = 74  (Rohit, match 3)
  team_scores[2, -1]  = 27  (Hardik, last match)

  team_scores[0, :]   = [ 72  45  88  31  64   0 103  52  19  77  41  93  28  66]
  (Virat's scores all season)

  team_scores[:, 0]   = [72 56 45]
  (All players' scores in match 1)

  team_scores[0:2, 0:4] =  (first 2 players, first 4 matches)
[[72 45 88 31]
 [56  0 74 88]]


In [ ]:
# Operations on 2D arrays
# axis=0 means 'collapse rows' (compute down each column)
# axis=1 means 'collapse columns' (compute across each row)

print("Operations across axes:")
print()

# Total runs per player (sum across all matches for each player = axis=1)
total_per_player = team_scores.sum(axis=1)
print(f"Total runs per player (axis=1):  {total_per_player}")
print(f"  Virat: {total_per_player[0]}, Rohit: {total_per_player[1]}, Hardik: {total_per_player[2]}")
print()

# Average score per match (mean across all players for each match = axis=0)
avg_per_match = team_scores.mean(axis=0).round(1)
print(f"Average score per match (axis=0): {avg_per_match}")
print()

# Highest score in each match
max_per_match = team_scores.max(axis=0)
print(f"Highest score per match:          {max_per_match}")
print()

# Each player's average
avg_per_player = team_scores.mean(axis=1).round(1)
print(f"Average per player:               {avg_per_player}")
players = ["Virat", "Rohit", "Hardik"]
for name, avg in zip(players, avg_per_player):
    print(f"  {name}: {avg}")

### Whiteboard: visualising axis

The `axis` parameter is one of the most confusing parts of NumPy for beginners. Here is a visual to anchor it:

```
team_scores shape: (3, 14)
                    |   |
                 axis=0  axis=1
                (rows)  (columns)

axis=0 — operations run DOWN each column (across rows)
         result shape: (14,) — one value per column

         [72, 45, 88, ...]   <- row 0 (Virat)
         [56,  0, 74, ...]   <- row 1 (Rohit)    } collapsed
         [45, 32,  0, ...]   <- row 2 (Hardik)   }
          --   --   --
         [sum sum sum ...]   <- one result per column

axis=1 — operations run ACROSS each row (across columns)
         result shape: (3,) — one value per row

         [72, 45, 88, 31, ...] -> sum -> 779  (Virat total)
         [56,  0, 74, 88, ...] -> sum -> 710  (Rohit total)
         [45, 32,  0, 67, ...] -> sum -> 580  (Hardik total)
```

Memory trick: **axis=0 collapses the first number in the shape tuple. axis=1 collapses the second.**

## 3D — Two seasons of data

In [ ]:
# 3D array: 2 seasons x 3 players x 8 matches
# Think of it as a stack of 2D arrays

two_seasons = np.array([
    # Season 1 (2023)
    [
        [72, 45, 88, 31, 64,  0, 103, 52],   # Virat
        [56,  0, 74, 88, 32, 45,  61, 17],   # Rohit
        [45, 32,  0, 67, 18, 54,  78, 22],   # Hardik
    ],
    # Season 2 (2024)
    [
        [65, 80, 44, 91, 33, 58,  22, 71],   # Virat
        [39, 55, 82,  0, 77, 21,  64, 48],   # Rohit
        [28, 44, 63, 37, 82,  0,  55, 71],   # Hardik
    ]
])

print(f"Shape: {two_seasons.shape}")
print(f"  {two_seasons.shape[0]} seasons")
print(f"  {two_seasons.shape[1]} players")
print(f"  {two_seasons.shape[2]} matches per season")
print(f"  Total scores: {two_seasons.size}")
print()

# 3D indexing: [season, player, match]
print("3D indexing examples:")
print(f"  two_seasons[0, 0, 0]  = {two_seasons[0, 0, 0]}  (Season 1, Virat, Match 1)")
print(f"  two_seasons[1, 1, 2]  = {two_seasons[1, 1, 2]}  (Season 2, Rohit, Match 3)")
print(f"  two_seasons[0, :, 0]  = {two_seasons[0, :, 0]}  (Season 1, all players, Match 1)")
print(f"  two_seasons[:, 0, :]  = Virat's scores in both seasons:")
print(two_seasons[:, 0, :])

Shape: (2, 3, 8)
  2 seasons
  3 players
  8 matches per season
  Total scores: 48

3D indexing examples:
  two_seasons[0, 0, 0]  = 72  (Season 1, Virat, Match 1)
  two_seasons[1, 1, 2]  = 82  (Season 2, Rohit, Match 3)
  two_seasons[0, :, 0]  = [72 56 45]  (Season 1, all players, Match 1)
  two_seasons[:, 0, :]  = Virat's scores in both seasons:
[[ 72  45  88  31  64   0 103  52]
 [ 65  80  44  91  33  58  22  71]]


## Reshaping arrays

One of the most useful NumPy operations is **reshape** — changing the shape of an array without changing its data. The total number of elements must stay the same.

In [ ]:
# Start with a 1D array of 12 match scores
scores_1d = np.array([72, 45, 88, 31, 64, 0, 103, 52, 19, 77, 41, 93])
print("Original 1D:", scores_1d)
print(f"Shape: {scores_1d.shape}")
print()

# Reshape into 2D: 3 players x 4 matches
scores_2d = scores_1d.reshape(3, 4)
print("Reshaped to (3, 4):")
print(scores_2d)
print(f"Shape: {scores_2d.shape}")
print()

# Reshape into 2D: 4 groups x 3 scores
scores_4x3 = scores_1d.reshape(4, 3)
print("Reshaped to (4, 3):")
print(scores_4x3)
print()

# Use -1 to let NumPy calculate one dimension automatically
scores_auto = scores_1d.reshape(6, -1)   # 6 rows, NumPy works out the columns
print("Reshaped to (6, -1) — NumPy auto-calculates columns:")
print(scores_auto)
print(f"Shape: {scores_auto.shape}")

---
### Exercise 3 — Working with 2D Arrays

**Task:** Use `team_scores` (shape `3 x 14`) to answer the following questions. Each answer should be written as a single NumPy expression — no loops.

In [ ]:
# Recall:
# team_scores row 0 = Virat
# team_scores row 1 = Rohit
# team_scores row 2 = Hardik
# Columns = matches 1-14

# Q1: Rohit's scores across all 14 matches (entire row 1)
rohit_all = team_scores[1,:]

# Q2: All three players' scores in match 5 (column index 4)
match5_scores = team_scores[:,4]

# Q3: Hardik's scores in matches 3 through 7 (row 2, columns 2 to 6)
hardik_mid = team_scores[2,2:7]

# Q4: The highest score any player hit in any single match
overall_max = team_scores.max

# Q5: Each player's highest score across the season
player_max = team_scores.max(axis=1)

# Q6: The match in which the team's combined score was highest
#     Hint: sum across players (axis=0), then use .argmax()
#     .argmax() returns the INDEX of the maximum value
best_match_index = team_scores.sum(axis=0).argmax()
best_match_number = best_match_index + 1   # convert 0-based index to match number


print(f"Q1 Rohit all:        {rohit_all}")
print(f"Q2 Match 5 scores:   {match5_scores}")
print(f"Q3 Hardik mid:       {hardik_mid}")
print(f"Q4 Overall max:      {overall_max}")
print(f"Q5 Player max:       {player_max}")
print(f"Q6 Best match:       Match {best_match_number}")
print()
print(f"Correct Q1 (14 scores):         {len(rohit_all) == 14}")
print(f"Correct Q2 (3 scores):          {len(match5_scores) == 3}")
print(f"Correct Q4 (max is 103):        {overall_max == 103}")
print(f"Correct Q5 (3 values):          {len(player_max) == 3}")

<details>
<summary>Show solution</summary>

```python
rohit_all        = team_scores[1, :]
match5_scores    = team_scores[:, 4]
hardik_mid       = team_scores[2, 2:7]
overall_max      = team_scores.max()
player_max       = team_scores.max(axis=1)
best_match_index = team_scores.sum(axis=0).argmax()
```

</details>

---
# Part 5 — Array Operations

NumPy operations are **vectorised** — they apply to every element without you writing a loop. This is one of the most important ideas in the session.

## Arithmetic operations

In [ ]:
virat_runs = np.array([72, 45, 88, 31, 64, 0, 103, 52, 19, 77, 41, 93, 28, 66])

# Scalar operations — applied to every element
print("Scalar operations:")
print(f"  virat_runs + 10    = {virat_runs + 10}")      # add 10 to each score
print(f"  virat_runs * 2     = {virat_runs * 2}")        # double each score
print(f"  virat_runs ** 2    = {virat_runs ** 2}")       # square each score
print(f"  virat_runs / 10    = {(virat_runs / 10).round(1)}")  # divide each
print()

# Array-to-array operations — element-wise (arrays must have the same shape)
rohit_runs = np.array([56, 0, 74, 88, 32, 45, 61, 17, 90, 38, 72, 5, 83, 49])

print("Array + Array (element-wise):")
combined = virat_runs + rohit_runs
print(f"  Partnership total per match: {combined}")
print()

diff = virat_runs - rohit_runs
print(f"  Virat minus Rohit per match: {diff}")

Scalar operations:
  virat_runs + 10    = [ 82  55  98  41  74  10 113  62  29  87  51 103  38  76]
  virat_runs * 2     = [144  90 176  62 128   0 206 104  38 154  82 186  56 132]
  virat_runs ** 2    = [ 5184  2025  7744   961  4096     0 10609  2704   361  5929  1681  8649
   784  4356]
  virat_runs / 10    = [ 7.2  4.5  8.8  3.1  6.4  0.  10.3  5.2  1.9  7.7  4.1  9.3  2.8  6.6]

Array + Array (element-wise):
  Partnership total per match: [128  45 162 119  96  45 164  69 109 115 113  98 111 115]

  Virat minus Rohit per match: [ 16  45  14 -57  32 -45  42  35 -71  39 -31  88 -55  17]


In [ ]:
# Boolean indexing — one of NumPy's most powerful features
# A comparison on an array returns a boolean array
# That boolean array can then be used as a filter

print("Boolean indexing:")
print()

# Step 1: the comparison returns a True/False array
above_50_mask = virat_runs > 50
print(f"virat_runs > 50 mask:")
print(above_50_mask)
print()

# Step 2: use the mask to filter the original array
half_centuries = virat_runs[above_50_mask]
print(f"Scores above 50:    {half_centuries}")
print(f"Count of 50+:       {len(half_centuries)}")
print()

# You can chain these in one line
ducks = virat_runs[virat_runs == 0]
print(f"Ducks (score == 0): {ducks}  ({len(ducks)} duck{'s' if len(ducks) != 1 else ''})")

# Combine conditions with & (and) and | (or)
between_30_60 = virat_runs[(virat_runs >= 30) & (virat_runs <= 60)]
print(f"Scores 30-60:       {between_30_60}")

In [ ]:
# Aggregate functions — the full set

print("Aggregate functions on virat_runs:")
print(f"  .sum()    = {virat_runs.sum()}         total runs")
print(f"  .mean()   = {virat_runs.mean():.2f}        batting average")
print(f"  .max()    = {virat_runs.max()}         highest score")
print(f"  .min()    = {virat_runs.min()}           lowest score")
print(f"  .std()    = {virat_runs.std():.2f}        standard deviation (consistency)")
print(f"  .argmax() = {virat_runs.argmax()}          index of highest score")
print(f"  .argmin() = {virat_runs.argmin()}          index of lowest score")
print(f"  .cumsum() = {virat_runs.cumsum()}")
print(f"             (running total after each match)")
print()

# Practical use of argmax: which match had the highest score?
best_match = virat_runs.argmax()
print(f"Virat's highest score ({virat_runs[best_match]}) was in match {best_match + 1}")

Aggregate functions on virat_runs:
  .sum()    = 779         total runs
  .mean()   = 55.64        batting average
  .max()    = 103         highest score
  .min()    = 0           lowest score
  .std()    = 28.95        standard deviation (consistency)
  .argmax() = 6          index of highest score
  .argmin() = 5          index of lowest score
  .cumsum() = [ 72 117 205 236 300 300 403 455 474 551 592 685 713 779]
             (running total after each match)

Virat's highest score (103) was in match 7


---
# Part 6 — Performance Comparison

## The question

We have been saying NumPy is faster than Python lists. By how much? Let's measure it properly.

We will compare three operations:
1. Summing a large array
2. Squaring every element
3. Finding all values above a threshold

We will do each with a Python list and with a NumPy array, and time them.

The dataset: IPL career statistics — 1,000,000 simulated ball-by-ball run records.

In [ ]:
# Generate 1 million simulated ball-by-ball run outcomes
# Each ball: 0 to 6 runs
N = 1_000_0000

np.random.seed(42)   # for reproducibility
runs_array = np.random.randint(0, 7, size=N)   # NumPy array
runs_list  = runs_array.tolist()               # Python list (same data)

print(f"Dataset: {N:,} ball-by-ball run outcomes")
print(f"NumPy array size: {runs_array.nbytes / 1024 / 1024:.1f} MB")
print(f"First 20 values:  {runs_array[:20]}")
print(f"\nValue distribution:")
for run_val in range(7):
    count = int((runs_array == run_val).sum())
    print(f"  {run_val} runs: {count:>8,} times ({count/N*100:.1f}%)")

Dataset: 10,000,000 ball-by-ball run outcomes
NumPy array size: 76.3 MB
First 20 values:  [6 3 4 6 2 4 4 6 1 2 6 2 2 4 3 2 5 4 1 3]

Value distribution:
  0 runs: 1,425,937 times (14.3%)
  1 runs: 1,429,561 times (14.3%)
  2 runs: 1,428,587 times (14.3%)
  3 runs: 1,429,199 times (14.3%)
  4 runs: 1,429,574 times (14.3%)
  5 runs: 1,428,606 times (14.3%)
  6 runs: 1,428,536 times (14.3%)


In [ ]:
# Test 1: Summing all 1 million values

REPEATS = 10   # run each test multiple times and take the average

# Python list sum
list_times = []
for _ in range(REPEATS):
    start = time.perf_counter()
    total = sum(runs_list)
    list_times.append(time.perf_counter() - start)
list_avg = sum(list_times) / REPEATS

# NumPy sum
numpy_times = []
for _ in range(REPEATS):
    start = time.perf_counter()
    total = runs_array.sum()
    numpy_times.append(time.perf_counter() - start)
numpy_avg = sum(numpy_times) / REPEATS

speedup = list_avg / numpy_avg

print("Test 1: Sum of 1,000,000 values")
print(f"  Python list:  {list_avg*1000:.2f} ms")
print(f"  NumPy array:  {numpy_avg*1000:.2f} ms")
print(f"  NumPy is {speedup:.0f}x faster")

Test 1: Sum of 1,000,000 values
  Python list:  76.94 ms
  NumPy array:  6.59 ms
  NumPy is 12x faster


In [ ]:
# Test 2: Square every element (1 million multiplications)

# Python list
list_times = []
for _ in range(REPEATS):
    start = time.perf_counter()
    squared = [x * x for x in runs_list]
    list_times.append(time.perf_counter() - start)
list_avg = sum(list_times) / REPEATS

# NumPy
numpy_times = []
for _ in range(REPEATS):
    start = time.perf_counter()
    squared = runs_array ** 2
    numpy_times.append(time.perf_counter() - start)
numpy_avg = sum(numpy_times) / REPEATS

speedup = list_avg / numpy_avg

print("Test 2: Square every element")
print(f"  Python list:  {list_avg*1000:.2f} ms")
print(f"  NumPy array:  {numpy_avg*1000:.2f} ms")
print(f"  NumPy is {speedup:.0f}x faster")

Test 2: Square every element
  Python list:  244.85 ms
  NumPy array:  30.36 ms
  NumPy is 8x faster


In [ ]:
# Test 3: Count how many values are above 3 (boundaries: 4, 5, or 6)

# Python list
list_times = []
for _ in range(REPEATS):
    start = time.perf_counter()
    count = sum(1 for x in runs_list if x > 3)
    list_times.append(time.perf_counter() - start)
list_avg = sum(list_times) / REPEATS

# NumPy
numpy_times = []
for _ in range(REPEATS):
    start = time.perf_counter()
    count = (runs_array > 3).sum()
    numpy_times.append(time.perf_counter() - start)
numpy_avg = sum(numpy_times) / REPEATS

speedup = list_avg / numpy_avg

print("Test 3: Count values above 3")
print(f"  Python list:  {list_avg*1000:.2f} ms")
print(f"  NumPy array:  {numpy_avg*1000:.2f} ms")
print(f"  NumPy is {speedup:.0f}x faster")

In [ ]:
# Summary table + memory comparison

import sys

# Memory usage
array_bytes = runs_array.nbytes
list_bytes  = sys.getsizeof(runs_list) + sum(sys.getsizeof(x) for x in runs_list[:100]) * (N // 100)

print("Memory usage for 1,000,000 integers:")
print(f"  Python list:  ~{list_bytes / 1024 / 1024:.0f} MB")
print(f"  NumPy array:     {array_bytes / 1024 / 1024:.0f} MB")
print(f"  NumPy uses ~{list_bytes // array_bytes}x less memory")
print()
print("Why?")
print("  Python list: each integer is a full Python object (28 bytes each)")
print("               plus the list stores a pointer (8 bytes) per element")
print("               total: ~36 bytes per element")
print("  NumPy array: each int64 is 8 bytes, stored contiguously")
print("               total: 8 bytes per element")

### Why does this matter in practice?

Today we ran 1 million values. A real dataset might have:
- 50 million rows of transaction data
- 100 million rows of sensor readings
- A 4K image: 3840 x 2160 x 3 = 24.9 million values

At that scale, the difference between a 30x speedup and writing loops is the difference between a 1-second result and a 30-second wait — or between a 1-minute job and a 30-minute job.

Pandas, which we will cover in the next module, is built entirely on top of NumPy. Every Pandas DataFrame column is a NumPy array. Everything you learned today is working underneath every Pandas operation you will run.

---
### Exercise 4 — Performance Measurement

**Task:** Complete the comparison below. You are measuring the time to compute the **running total** (cumulative sum) of 1 million values using both approaches.

- Python list version: use a manual loop that builds a new list
- NumPy version: use `.cumsum()`

Then answer the two questions at the bottom.

In [ ]:
# Python list cumulative sum
start = time.perf_counter()

running_total = []
cumulative    = 0
for x in runs_list:
    cumulative += x
    running_total.append(cumulative)

list_time = time.perf_counter() - start


# NumPy cumulative sum
start = time.perf_counter()

numpy_cumsum = runs_array.cumsum()   # <-- which NumPy method?

numpy_time = time.perf_counter() - start


print(f"Python list cumsum: {list_time*1000:.2f} ms")
print(f"NumPy cumsum:       {numpy_time*1000:.2f} ms")
print(f"Speedup:            {list_time / numpy_time:.0f}x")
print()

# Q1: What is the total runs scored across all 1 million balls?
#     (It is the last value in the cumulative sum)
total_runs = numpy_cumsum[-1]   # <-- which index gives the last element?
print(f"Q1 Total runs across all 1M balls: {total_runs:,}")

# Q2: After how many balls had 1,000,000 cumulative runs been scored?
#     Use np.searchsorted() — it finds the first index where the value crosses a threshold
ball_number = np.searchsorted(numpy_cumsum, 1_000_000)
print(f"Q2 Ball number when 1M runs were reached: {ball_number:,}")
print()
print(f"Correct (total > 2.5M): {total_runs > 2_500_000}")
print(f"Correct (ball < 500k):  {ball_number < 500_000}")

<details>
<summary>Show solution</summary>

```python
numpy_cumsum = runs_array.cumsum()
total_runs   = numpy_cumsum[-1]
```

</details>

---
## Final Exercise — End-to-End Challenge

This exercise uses everything from today. You are the analyst for an IPL franchise. The coaching staff has sent you a 2D array of match scores and wants a full performance report — no loops allowed anywhere.

**The data:** 5 players x 14 matches.

In [ ]:
np.random.seed(7)

# Full squad: 5 batters, 14 matches each
squad_scores = np.array([
    [72, 45, 88, 31, 64,  0, 103, 52, 19, 77, 41, 93, 28, 66],   # Virat
    [56,  0, 74, 88, 32, 45,  61, 17, 90, 38, 72,  5, 83, 49],   # Rohit
    [45, 32,  0, 67, 18, 54,  78, 22, 41, 33, 60, 15, 88, 27],   # Hardik
    [33, 61, 45,  0, 77, 83,  22, 58, 44, 71,  0, 66, 31, 52],   # Shubman
    [88, 14, 57, 72,  0, 38,  91, 43, 66, 22, 84,  0, 47, 73],   # Shreyas
])

PLAYERS = ["Virat", "Rohit", "Hardik", "Shubman", "Shreyas"]

print(f"squad_scores shape: {squad_scores.shape}")
print()

# --- YOUR ANSWERS BELOW --- all using NumPy, no explicit loops ---

# Q1: Season total runs for each player
season_totals = squad_scores.sum(axis=1)   # <-- which axis?

# Q2: Batting average for each player
batting_avgs = squad_scores.mean(axis=1)    # <-- which method and axis?

# Q3: Highest individual score each player hit this season
season_best = squad_scores.max(axis=1)     # <-- which method and axis?

# Q4: Number of ducks (score == 0) each player had
#     Hint: create a boolean array (squad_scores == 0) then sum along axis=1
duck_counts = (squad_scores == 0).sum(axis=1)

# Q5: Number of half-centuries (50+) each player hit
fifties = (squad_scores >= 50).sum(axis=1)

# Q6: Which player has the highest season total?
#     Hint: use .argmax() on season_totals
top_scorer_index = season_totals.argmax()       # <-- which method?
top_scorer_name  = PLAYERS[top_scorer_index]

# Q7: Which match had the highest total team score?
match_totals     = squad_scores.sum(axis=0)
best_match_index = match_totals.argsum()        # <-- which method?
best_match_num   = best_match_index + 1

# Q8: Reshape squad_scores into a 3D array (seasons x players x matches)
#     Pretend the 14 matches are split into 2 halves of 7 matches each
#     Result shape should be (2, 5, 7)
half_seasons = squad_scores.reshape(2,5,7)   # <-- 3 numbers


# --- PRINT THE REPORT ---
print("FRANCHISE BATTING REPORT")
print("=" * 65)
print(f"  {'Player':<10} {'Total':>7}  {'Average':>8}  {'Best':>6}  {'Ducks':>6}  {'50+':>5}")
print("  " + "-" * 55)

for i, player in enumerate(PLAYERS):
    print(f"  {player:<10} {season_totals[i]:>7}  {batting_avgs[i]:>8.1f}  {season_best[i]:>6}  {duck_counts[i]:>6}  {fifties[i]:>5}")

print()
print(f"Top scorer: {top_scorer_name} ({season_totals[top_scorer_index]} runs)")
print(f"Best team match: Match {best_match_num} (combined score: {match_totals[best_match_index]})")
print(f"Half-seasons shape: {half_seasons.shape}")
print()
print(f"Correct (shape 5x14):          {squad_scores.shape == (5, 14)}")
print(f"Correct (5 totals):            {len(season_totals) == 5}")
print(f"Correct (half shape 2x5x7):    {half_seasons.shape == (2, 5, 7)}")

<details>
<summary>Show solution</summary>

```python
season_totals    = squad_scores.sum(axis=1)
batting_avgs     = squad_scores.mean(axis=1)
season_best      = squad_scores.max(axis=1)
duck_counts      = (squad_scores == 0).sum(axis=1)
fifties          = (squad_scores >= 50).sum(axis=1)
top_scorer_index = season_totals.argmax()
match_totals     = squad_scores.sum(axis=0)
best_match_index = match_totals.argmax()
half_seasons     = squad_scores.reshape(2, 5, 7)
```

</details>

---
## Session Summary

### Why NumPy?
Python lists store scattered Python objects connected by pointers. NumPy arrays store values contiguously in memory, all of the same type. This makes bulk numeric operations 10x–100x faster and uses far less memory.

### Creating arrays

| Function | Use case |
|---|---|
| `np.array([...])` | From existing data |
| `np.zeros(n)` | Pre-allocated empty slots |
| `np.ones(n)` | All ones |
| `np.arange(start, stop)` | Integer sequence (like `range`) |
| `np.linspace(start, stop, n)` | N evenly spaced floats |
| `np.full(n, value)` | All elements the same value |

### Array properties

| Property | Meaning |
|---|---|
| `.shape` | Dimensions as a tuple — the most important one |
| `.ndim` | Number of dimensions (length of `.shape`) |
| `.size` | Total number of elements (product of `.shape`) |
| `.dtype` | Type of each element (`int64`, `float64`, `bool`) |
| `.nbytes` | Memory used |

### N-dimensional arrays
- **1D**: a sequence — `shape (n,)`
- **2D**: a table of rows and columns — `shape (rows, cols)` — index with `[row, col]`
- **3D**: a stack of tables — `shape (depth, rows, cols)` — index with `[d, r, c]`
- `:` means "all elements in this dimension" — use it in slices
- `axis=0` collapses rows (operates down columns); `axis=1` collapses columns (operates across rows)

### Key operations
- Arithmetic (`+`, `-`, `*`, `**`) applies element-wise with no loop
- Boolean indexing: `arr[arr > 50]` returns only the matching elements
- Aggregates: `.sum()`, `.mean()`, `.max()`, `.min()`, `.std()`, `.argmax()`, `.cumsum()`
- `.reshape(a, b)` changes shape without changing data — total elements must stay the same

---

### What comes next

**Session 5.2** introduces **Pandas** — the library built on top of NumPy that adds named columns, mixed data types, and powerful tools for loading, cleaning, and analysing real datasets. Every Pandas Series is a NumPy array with a label. Everything you learned today is working underneath every Pandas operation.

---

### Quick reference

```python
import numpy as np

# Create
a = np.array([1, 2, 3])           # from list
b = np.zeros((3, 4))              # 3x4 of zeros
c = np.arange(1, 15)              # 1 to 14
d = np.linspace(0, 1, 5)          # 5 points 0.0 to 1.0

# Properties
a.shape   # (3,)
a.ndim    # 1
a.size    # 3
a.dtype   # int64

# Index and slice (2D)
b[0, 1]      # row 0, col 1
b[1, :]      # all of row 1
b[:, 2]      # all of col 2
b[0:2, 1:3]  # sub-matrix

# Operations
a * 2              # element-wise
a[a > 1]           # boolean filter
a.sum(axis=1)      # sum across columns
a.mean(axis=0)     # mean down rows
a.reshape(1, 3)    # change shape
```